In [15]:
import numpy as np 
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AffinityPropagation
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")
import plotly as py
import plotly.graph_objs as go
import os
py.offline.init_notebook_mode(connected = True)
#print(os.listdir("../input"))
import datetime as dt
import missingno as msno
plt.rcParams['figure.dpi'] = 140

In [17]:
file_path = r'C:\Users\jayar\Downloads\project\Netflix-Content-Strat\Week-1-task\netflix_titles (1).csv'

df = pd.read_csv(file_path)
df.head(3)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...


In [18]:
print(f"Number of rows: {len(df)}")

Number of rows: 8807


In [19]:
# Fill null values: mean for numerical, mode for categorical columns
print("Filling null values:")
print(f"Remaining null values before filling:\n{df.isnull().sum()}\n")

# Identify numerical and categorical columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical columns: {numerical_cols}")
print(f"Categorical columns: {categorical_cols}\n")

# Fill numerical columns with mean
for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        mean_value = df[col].mean()
        df[col] = df[col].fillna(mean_value)
        print(f"✓ Filled '{col}' (numerical) with mean: {mean_value:.2f}")

# Fill categorical columns with mode
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_value = df[col].dropna().mode()
        if len(mode_value) > 0:
            mode_value = mode_value[0]
            df[col] = df[col].fillna(mode_value)
            print(f"✓ Filled '{col}' (categorical) with mode: '{mode_value}'")

print(f"\n{'='*50}")
print(f"Null values after filling:\n{df.isnull().sum()}")
print(f"{'='*50}")
print(f"Dataset shape: {df.shape}")
print("\n✓ All missing values have been successfully handled!")


Filling null values:
Remaining null values before filling:
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

Numerical columns: ['release_year']
Categorical columns: ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'rating', 'duration', 'listed_in', 'description']

✓ Filled 'director' (categorical) with mode: 'Rajiv Chilaka'
✓ Filled 'cast' (categorical) with mode: 'David Attenborough'
✓ Filled 'country' (categorical) with mode: 'United States'
✓ Filled 'date_added' (categorical) with mode: 'January 1, 2020'
✓ Filled 'rating' (categorical) with mode: 'TV-MA'
✓ Filled 'duration' (categorical) with mode: '1 Season'

Null values after filling:
show_id         0
type            0
title           0
director        0
cast            0
country         0
da

In [20]:
# Check for duplicates
print("Duplicate rows:")
print(df.duplicated().sum())

# Drop duplicates, keeping the first occurrence
df = df.drop_duplicates(keep='first')

print(f"\nDataset shape after removing duplicates: {df.shape}")
print(f"Total duplicates removed: {len(df) - df.shape[0]}")


Duplicate rows:
0

Dataset shape after removing duplicates: (8807, 12)
Total duplicates removed: 0


In [21]:
# Check for inconsistent categories in the 'type' column
print("Unique values in 'type' column:")
print(df['type'].unique())
print(f"\nValue counts:\n{df['type'].value_counts()}")

# Standardize the 'type' column (fix inconsistencies like 'Tv Show' vs 'TV Show')
df['type'] = df['type'].str.replace('Tv Show', 'TV Show', case=False)

print(f"\nAfter standardization:")
print(df['type'].value_counts())

# Check for other categorical columns with potential inconsistencies
print("\n\nChecking 'rating' column:")
print(f"Unique ratings: {df['rating'].nunique()}")
print(f"Ratings:\n{df['rating'].value_counts()}")
# Check for inconsistencies in other categorical columns
print("\n\nChecking other categorical columns for inconsistencies:")
for col in ['rating', 'country', 'listed_in']:
    if col in df.columns:
        print(f"\n{col.upper()}:")
        print(f"Unique values: {df[col].nunique()}")
        print(df[col].value_counts().head())

# Standardize whitespace issues (leading/trailing spaces)
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
print("\n\nData cleaning complete!")
print(f"Final dataset shape: {df.shape}")
# Display summary of rows affected by data cleaning for inconsistencies in category
print("\nSummary of data cleaning:")
print(f"- Rows removed due to missing content: {8807 - len(df)}")


Unique values in 'type' column:
<StringArray>
['Movie', 'TV Show']
Length: 2, dtype: str

Value counts:
type
Movie      6131
TV Show    2676
Name: count, dtype: int64

After standardization:
type
Movie      6131
TV Show    2676
Name: count, dtype: int64


Checking 'rating' column:
Unique ratings: 17
Ratings:
rating
TV-MA       3211
TV-14       2160
TV-PG        863
R            799
PG-13        490
TV-Y7        334
TV-Y         307
PG           287
TV-G         220
NR            80
G             41
TV-Y7-FV       6
NC-17          3
UR             3
74 min         1
84 min         1
66 min         1
Name: count, dtype: int64


Checking other categorical columns for inconsistencies:

RATING:
Unique values: 17
rating
TV-MA    3211
TV-14    2160
TV-PG     863
R         799
PG-13     490
Name: count, dtype: int64

COUNTRY:
Unique values: 748
country
United States     3649
India              972
United Kingdom     419
Japan              245
South Korea        199
Name: count, dtype: int64

L

In [ ]:
# Display the entire cleaned dataset
print("Cleaned Netflix Dataset:")
print("="*100)
print(df)
print("\n" + "="*100)
print(f"\nDataset Info:")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nBasic Statistics:\n{df.describe()}")